In [1]:
import os
import re
import warnings

from scipy.signal import welch
from scipy.stats import entropy, skew, kurtosis
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import time
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [2]:
warnings.filterwarnings('ignore')

In [3]:
# Analyzing dataset
dataset = "./Dataset"
dataset_actions = {}

for root, dirs, files in os.walk(dataset):
    for file in files:
        file = file[:-4]
        action =  re.sub(r'\d+$', '', file)
        if action not in dataset_actions:
            dataset_actions[action] = 1
        else:
            dataset_actions[action] += 1

Dataset_actions_log = "Dataset Actions Count\n"
total = 0
for key, value in dataset_actions.items():
    total += value
    Dataset_actions_log += f"{key}: {value} | "
Dataset_actions_log += f"total: {total}"
print(Dataset_actions_log)

Dataset Actions Count
final: 175 | grenade: 195 | idle: 21 | reload: 172 | shield: 195 | total: 758


In [4]:
# Read in dataset into a dataframe of 7 columns, 6 measurement columns, 1 label
# each row has 6 lists, 1 label corresponding to 1 file
dataset = "./Dataset"
dataset_pd = pd.DataFrame(columns=["x_acc", "y_acc", "z_acc", "x_gyro", "y_gyro", "z_gyro", "action"])

for root, dirs, files in os.walk(dataset):
    for file in files:
        if 'idle' not in file:
            file_path = os.path.join(root, file)
            file_pd = pd.read_csv(file_path)
            x_acc = file_pd.iloc[:, 0].tolist()
            y_acc = file_pd.iloc[:, 1].tolist()
            z_acc = file_pd.iloc[:, 2].tolist()
            x_gyro = file_pd.iloc[:, 3].tolist()
            y_gyro = file_pd.iloc[:, 4].tolist()
            z_gyro = file_pd.iloc[:, 5].tolist()
            file = file[:-4]
            action =  re.sub(r'\d+$', '', file)
            row = {"x_acc": x_acc, "y_acc": y_acc, "z_acc": z_acc, "x_gyro": x_gyro, "y_gyro": y_gyro, "z_gyro": z_gyro, "action": action}
            dataset_pd = pd.concat([dataset_pd, pd.DataFrame([row])], ignore_index=True)
        else: 
            for chunk_idle in pd.read_csv(os.path.join(root, file), chunksize=200): # take idle at 200 row max length
                x_acc = chunk_idle.iloc[:, 0].tolist()
                y_acc = chunk_idle.iloc[:, 1].tolist()
                z_acc = chunk_idle.iloc[:, 2].tolist()
                x_gyro = chunk_idle.iloc[:, 3].tolist()
                y_gyro = chunk_idle.iloc[:, 4].tolist()
                z_gyro = chunk_idle.iloc[:, 5].tolist()
                file = file[:-4]
                action =  re.sub(r'\d+$', '', file)
                row = {"x_acc": x_acc, "y_acc": y_acc, "z_acc": z_acc, "x_gyro": x_gyro, "y_gyro": y_gyro, "z_gyro": z_gyro, "action": 'idle'}
                dataset_pd = pd.concat([dataset_pd, pd.DataFrame([row])], ignore_index=True)
# df 
# [] [] [] [] [] [] label

In [ ]:
# Code to Extract all features
# able to extract the following features
# time domain: mean, std, rms, min, max, median, 25th percentile, 50th percentile, 75th percentile, kurtosis, skew 
# zero-crossing rate, peak count
# frequency domain: mean, max, Spectral Entropy, total power, Spectral centroid, DC component, 
# Dominant frequency, Power Spectral Density, average energy per coefficient
"""
all_features = pd.DataFrame()
for i in range(len(dataset_pd.index)):
    data_dict = {}
    label = dataset_pd.iloc[i, len(dataset_pd.columns)-1]
    for j in range(len(dataset_pd.columns)-1):
        col_name = dataset_pd.columns[j]
        data = dataset_pd.iloc[i, j]
        # Compute time-domain features
        t_mean = np.mean(data)
        t_std_deviation = np.std(data)
        t_rms = np.sqrt(np.mean(np.square(data)))
        t_min = np.min(data)
        t_max = np.max(data)
        t_median = np.median(data)
        t_25_q = np.percentile(data, 25)
        t_50_q = np.percentile(data, 50)
        t_75_q = np.percentile(data, 75)
        t_zero_crossing = np.sum(np.diff(np.sign(data)) != 0)
        t_peaks = len(np.where((np.diff(np.sign(np.diff(data)))) < 0)[0])
        t_skew = skew(data)
        t_kurtosis = kurtosis(data)

        # Compute frequency-domain features
        freq_domain = np.fft.rfft(data)
        f_mean = abs(np.mean(freq_domain))
        f_max = abs(max(freq_domain))
        f_spectral_entropy = entropy(np.abs(freq_domain))
        f_total_power = np.sum(np.abs(freq_domain) ** 2)
        f_spectral_centroid = np.sum(np.arange(len(freq_domain)) * np.abs(freq_domain)) / np.sum(np.abs(freq_domain))
        f_DC_component = abs(freq_domain[0])
        f_dominant_frequency = np.argmax(np.abs(freq_domain))
        f_PSD = welch(data)[1]
        f_average_E_per_coeff = f_total_power / len(freq_domain)

        # Create dictionary
        data_dict.update({
            f"t_mean_{col_name}": t_mean,
            f"t_std_deviation_{col_name}": t_std_deviation,
            f"t_rms_{col_name}": t_rms,
            f"t_min_{col_name}": t_min,
            f"t_max_{col_name}": t_max,
            f"t_median_{col_name}": t_median,
            f"t_25_q_{col_name}": t_25_q,
            f"t_50_q_{col_name}": t_50_q,
            f"t_75_q_{col_name}": t_75_q,
            f"t_skew_{col_name}": t_skew,
            f"t_kurtosis_{col_name}": t_kurtosis,
            f"t_zero_crossing_{col_name}": t_zero_crossing,
            f"t_peaks_{col_name}": t_peaks,
            "freq_domain": freq_domain,
            f"f_mean_{col_name}": f_mean,
            f"f_max_{col_name}": f_max,
            f"f_spectral_entropy_{col_name}": f_spectral_entropy,
            f"f_total_power_{col_name}": f_total_power,
            f"f_spectral_centroid_{col_name}": f_spectral_centroid,
            "f_DC_component": f_DC_component,
            f"f_dominant_frequency_{col_name}": f_dominant_frequency,
            "f_PSD": f_PSD,
            f"f_average_E_per_coeff_{col_name}": f_average_E_per_coeff,
        })
    data_dict.update({f"label": label})
    
    all_features = pd.concat([all_features, pd.DataFrame([data_dict])], ignore_index=True)

all_features.to_csv("extracted_feature_with_label.csv", index=False)

label_encoder = LabelEncoder()
all_features['label'] = label_encoder.fit_transform(all_features['label'])
for label, encoded_label in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{label}: {encoded_label}")

print(all_features.shape)
"""

In [7]:
# Test 1: I will choose 8/9
# Time domain: mean, skew, kurtosis, rms, std
# Frequency domain: Average Energy per coefficient, mean, iqr
# Actually why the 3 gyro need frequency domain lmao 

max_time = 0

all_features = pd.DataFrame()
for i in range(len(dataset_pd.index)):
    start_time = time.time()

    data_dict = {}
    label = dataset_pd.iloc[i, len(dataset_pd.columns)-1]
    for j in range(len(dataset_pd.columns)-1):
        col_name = dataset_pd.columns[j]
        data = dataset_pd.iloc[i, j]
        # Compute time-domain features
        t_mean = np.mean(data)
        t_std_deviation = np.std(data)
        t_rms = np.sqrt(np.mean(np.square(data)))
        t_skew = skew(data)
        t_kurtosis = kurtosis(data)

        # Compute frequency-domain features
        freq_domain = np.fft.rfft(data)
        f_mean = abs(np.mean(freq_domain))
        f_max = abs(max(freq_domain))
        f_total_power = np.sum(np.abs(freq_domain) ** 2)
        f_average_E_per_coeff = f_total_power / len(freq_domain)

        # Create dictionary
        data_dict.update({
            f"t_mean_{col_name}": t_mean,
            f"t_std_deviation_{col_name}": t_std_deviation,
            f"t_rms_{col_name}": t_rms,
            f"t_skew_{col_name}": t_skew,
            f"t_kurtosis_{col_name}": t_kurtosis,
            f"f_mean_{col_name}": f_mean,
            f"f_max_{col_name}": f_max,
            f"f_total_power_{col_name}": f_total_power,
            f"f_average_E_per_coeff_{col_name}": f_average_E_per_coeff,
        })
    data_dict.update({f"label": label})
    
    all_features = pd.concat([all_features, pd.DataFrame([data_dict])], ignore_index=True)

# L2 Normalisation for each column
all_features = all_features / np.linalg.norm(all_features, axis=0)
# Scale to same range
scaler = StandardScaler()
scaled_data = scaler.fit_transform(all_features)
min_max_scaler = MinMaxScaler(feature_range=(0, 1))
scaled_min_max_data = min_max_scaler.fit_transform(scaled_data)
end_time = time.time()
elapsed_time = end_time - start_time
if elapsed_time > max_time:
    max_time = elapsed_time
print(f"Elapsed time: {elapsed_time} seconds")

print(f"Maxed time: {max_time} seconds.")


all_features.to_csv("extracted_feature_with_label.csv", index=False)

label_encoder = LabelEncoder()
all_features['label'] = label_encoder.fit_transform(all_features['label'])
for label, encoded_label in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{label}: {encoded_label}")

print(all_features.shape)

TypeError: loop of ufunc does not support argument 46170 of type str which has no callable conjugate method

In [8]:
# Initialize max_time
max_time = 0

# Initialize DataFrame to store features
all_features = pd.DataFrame()

# Iterate over dataset rows
for i in range(len(dataset_pd.index)):
    start_time = time.time()

    data_dict = {}
    label = dataset_pd.iloc[i, -1]  # Assuming label is in the last column
    for j in range(len(dataset_pd.columns) - 1):  # Exclude label column
        col_name = dataset_pd.columns[j]
        data = dataset_pd.iloc[i, j]

        # Compute time-domain features
        t_mean = np.mean(data)
        t_std_deviation = np.std(data)
        t_rms = np.sqrt(np.mean(np.square(data)))
        t_skew = skew(data)
        t_kurtosis = kurtosis(data)

        # Compute frequency-domain features
        freq_domain = np.fft.rfft(data)
        f_mean = abs(np.mean(freq_domain))
        f_max = abs(max(freq_domain))
        f_total_power = np.sum(np.abs(freq_domain) ** 2)
        f_average_E_per_coeff = f_total_power / len(freq_domain)

        # Update data dictionary
        data_dict.update({
            f"t_mean_{col_name}": t_mean,
            f"t_std_deviation_{col_name}": t_std_deviation,
            f"t_rms_{col_name}": t_rms,
            f"t_skew_{col_name}": t_skew,
            f"t_kurtosis_{col_name}": t_kurtosis,
            f"f_mean_{col_name}": f_mean,
            f"f_max_{col_name}": f_max,
            f"f_total_power_{col_name}": f_total_power,
            f"f_average_E_per_coeff_{col_name}": f_average_E_per_coeff,
        })
    print(time.time() - start_time)

    # Update data dictionary with label
    data_dict.update({"label": label})

    # Concatenate data_dict to all_features DataFrame
    all_features = pd.concat([all_features, pd.DataFrame([data_dict])], ignore_index=True)

    end_time = time.time()
    elapsed_time = end_time - start_time
    if elapsed_time > max_time:
        max_time = elapsed_time

# L2 Normalization for each feature column
all_features.iloc[:, :-1] = all_features.iloc[:, :-1].apply(lambda x: x / np.linalg.norm(x))

# Scale to same range
scaler = StandardScaler()
scaled_data = scaler.fit_transform(all_features.iloc[:, :-1])  # Exclude label column

min_max_scaler = MinMaxScaler(feature_range=(0, 1))
scaled_min_max_data = min_max_scaler.fit_transform(scaled_data)

print(f"Maxed time: {max_time} seconds.")

# Save extracted features with labels to CSV
all_features.to_csv("extracted_feature_with_label.csv", index=False)

# Encode labels
label_encoder = LabelEncoder()
all_features['label'] = label_encoder.fit_transform(all_features['label'])
for label, encoded_label in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{label}: {encoded_label}")

print(all_features.shape)


Maxed time: 0.010970354080200195 seconds.
final: 0
grenade: 1
idle: 2
reload: 3
shield: 4
(855, 55)
